# TP Máquina de Turing Universal
---

**¿Por qué se dice que una MTU es capaz de simular cualquier Máquina de Turing?**

**Suponer que se tiene una MT M que acepta todas las cadenas que terminan en 01. Indicar qué debería hacer una MTU con las siguientes entradas. Explicar en cada caso si la MTU acepta o rechaza y por qué**
* U(⟨M⟩,1101)
* U(⟨M⟩,100)
* U(⟨M⟩,01)
* U(⟨M⟩,111)

**Dada la siguiente MT M:** 

| Q   | 0        | 1        |
| --  | --       | --       |
| q0  | (q1,1,R) | (q1,0,R) |
| q1  | (qf,0,R) | (qf,1,R) |
| qf  | -        | -        |

1. Explicar que hace M 
2. Explicar qué información debería recibir una MTU para poder simular M
3. Codificar la cintar de MTU sabiendo que configuración de la cinta de MT M es 1 q0 0 1 1

1. **Codificación de una máquina simple**  
   * Definir una máquina $M$ que ...
   * Codificar sus estados, símbolos y transiciones en forma numérica

![alt text](image-1.png) ![alt text](image.png)
| Estado     | Función                     | Codificación |
| ---------- | --------------------------- | -----------: |
| `q0`       | Buscar un 0                 |        `000` |
| `q1`       | Buscar un 1                 |        `001` |
| `q2`       | Volver al comienzo          |        `010` |
| `q3`       | Verificar que solo queden Y |        `011` |
| `q_accept` | Aceptar                     |        `100` |
| `q_reject` | Rechazar                    |        `101` |

| Símbolo | Binario |       
| ------- | ------: |
| B       |     000 |
| 0       |     001 |
| 1       |     010 |
| X       |     011 |
| Y       |     100 |

| Movimiento | Binario |
| ---------- | ------: |
| L          |      00 |
| R          |      01 |
| S          |      10 |

| Transición original | Cadena binaria |
|---|---|
| δ(q0,X)=(q0,X,R) | `00001100001101` |
| δ(q0,0)=(q1,X,R) | `00000100101101` |
| δ(q0,Y)=(q3,Y,R) | `00010001110001` |
| δ(q0,1)=(q_reject,1,S) | `00001010101010` |
| δ(q0,B)=(q_accept,B,S) | `00000010000010` |
| δ(q1,0)=(q1,0,R) | `00100100100101` |
| δ(q1,X)=(q1,X,R) | `00101100101101` |
| δ(q1,Y)=(q1,Y,R) | `00110000110001` |
| δ(q1,1)=(q2,Y,L) | `00101001010000` |
| δ(q1,B)=(q_reject,B,S) | `00100010100010` |
| δ(q2,0)=(q2,0,L) | `01000101000100` |
| δ(q2,1)=(q2,1,L) | `01001001001000` |
| δ(q2,X)=(q2,X,L) | `01001101001100` |
| δ(q2,Y)=(q2,Y,L) | `01010001010000` |
| δ(q2,B)=(q0,B,R) | `01000000000001` |
| δ(q3,Y)=(q3,Y,R) | `01110001110001` |
| δ(q3,B)=(q_accept,B,S) | `01100010000010` |
| δ(q3,0)=(q_reject,0,S) | `01100110100110` |
| δ(q3,1)=(q_reject,1,S) | `01101010101010` |




2. **Simulación básica**  
   * Implementar en Python un programa que reciba:
     * La codificación de una máquina $M$
     * Una cadena de entrada $w$
   * El programa debe simular paso a paso la ejecución de $M$ sobre $w$

In [2]:
# ============================================================
# MAQUINA DE TURING UNIVERSAL
# Simula una MT que reconoce:
# L = { 0^n 1^n | n >= 0 }
# ============================================================


# ------------------------------------------------------------
# CODIFICACION DE ESTADOS
# ------------------------------------------------------------
#
# q0       = 000 = 0
# q1       = 001 = 1
# q2       = 010 = 2
# q3       = 011 = 3
# q_accept = 100 = 4
# q_reject = 101 = 5


ESTADOS_BIN = {
    0: "000",
    1: "001",
    2: "010",
    3: "011",
    4: "100",
    5: "101"
}


# ------------------------------------------------------------
# CODIFICACION DE SIMBOLOS
# ------------------------------------------------------------
#
# B = 000 = 0
# 0 = 001 = 1
# 1 = 010 = 2
# X = 011 = 3
# Y = 100 = 4
# ------------------------------------------------------------

SIMBOLOS = {
    "B": 0,
    "0": 1,
    "1": 2,
    "X": 3,
    "Y": 4
}

SIMBOLOS_INV = {
    0: "B",
    1: "0",
    2: "1",
    3: "X",
    4: "Y"
}

SIMBOLOS_BIN = {
    0: "000",
    1: "001",
    2: "010",
    3: "011",
    4: "100"
}


# ------------------------------------------------------------
# CODIFICACION DE MOVIMIENTOS
# ------------------------------------------------------------
#
# L = 00 = 0
# R = 01 = 1
# S = 10 = 2
# ------------------------------------------------------------

MOVIMIENTOS = {
    0: "L",
    1: "R",
    2: "S"
}

MOVIMIENTOS_BIN = {
    0: "00",
    1: "01",
    2: "10"
}


maquina = [

    # -------------------------
    # Estado q0
    # -------------------------

    # δ(q0,X) = (q0,X,R)
    (0, 3, 0, 3, 1),

    # δ(q0,0) = (q1,X,R)
    (0, 1, 1, 3, 1),

    # δ(q0,Y) = (q3,Y,R)
    (0, 4, 3, 4, 1),

    # δ(q0,1) = (q_reject,1,S)
    (0, 2, 5, 2, 2),

    # δ(q0,B) = (q_accept,B,S)
    (0, 0, 4, 0, 2),


    # -------------------------
    # Estado q1
    # -------------------------

    # δ(q1,0) = (q1,0,R)
    (1, 1, 1, 1, 1),

    # δ(q1,X) = (q1,X,R)
    (1, 3, 1, 3, 1),

    # δ(q1,Y) = (q1,Y,R)
    (1, 4, 1, 4, 1),

    # δ(q1,1) = (q2,Y,L)
    (1, 2, 2, 4, 0),

    # δ(q1,B) = (q_reject,B,S)
    (1, 0, 5, 0, 2),


    # -------------------------
    # Estado q2
    # -------------------------

    # δ(q2,0) = (q2,0,L)
    (2, 1, 2, 1, 0),

    # δ(q2,1) = (q2,1,L)
    (2, 2, 2, 2, 0),

    # δ(q2,X) = (q2,X,L)
    (2, 3, 2, 3, 0),

    # δ(q2,Y) = (q2,Y,L)
    (2, 4, 2, 4, 0),

    # δ(q2,B) = (q0,B,R)
    (2, 0, 0, 0, 1),


    # -------------------------
    # Estado q3
    # -------------------------

    # δ(q3,Y) = (q3,Y,R)
    (3, 4, 3, 4, 1),

    # δ(q3,B) = (q_accept,B,S)
    (3, 0, 4, 0, 2),

    # δ(q3,0) = (q_reject,0,S)
    (3, 1, 5, 1, 2),

    # δ(q3,1) = (q_reject,1,S)
    (3, 2, 5, 2, 2)
]


# ------------------------------------------------------------
# BUSCAR TRANSICION
# ------------------------------------------------------------

def buscar_transicion(maquina, estado, simbolo):

    for transicion in maquina:

        estado_actual = transicion[0]
        simbolo_leido = transicion[1]

        if estado_actual == estado and simbolo_leido == simbolo:
            return transicion

    return None


# ------------------------------------------------------------
# MOSTRAR CINTA
# ------------------------------------------------------------

def mostrar_cinta(cinta, cabezal, estado):

    posiciones = list(cinta.keys())

    minimo = min(posiciones)
    maximo = max(posiciones)

    print(f"\nEstado actual: q{estado} ({ESTADOS_BIN[estado]})")

    print("Cinta: ", end="")

    for posicion in range(minimo, maximo + 1):

        simbolo = SIMBOLOS_INV[cinta.get(posicion, 0)]

        if posicion == cabezal:
            print(f"[{simbolo}]", end=" ")
        else:
            print(f" {simbolo} ", end=" ")

    print()


# ------------------------------------------------------------
# MOSTRAR CODIFICACION BINARIA DE UNA TRANSICION
# ------------------------------------------------------------

def mostrar_codigo_binario(transicion):

    estado_actual = transicion[0]
    simbolo_leido = transicion[1]
    nuevo_estado = transicion[2]
    simbolo_escrito = transicion[3]
    movimiento = transicion[4]

    codigo = (
        ESTADOS_BIN[estado_actual]
        + SIMBOLOS_BIN[simbolo_leido]
        + ESTADOS_BIN[nuevo_estado]
        + SIMBOLOS_BIN[simbolo_escrito]
        + MOVIMIENTOS_BIN[movimiento]
    )

    return codigo


# ------------------------------------------------------------
# MAQUINA DE TURING UNIVERSAL
# ------------------------------------------------------------

def mtu(maquina, palabra, max_pasos=1000):

    # Validar palabra
    for simbolo in palabra:

        if simbolo not in ["0", "1"]:
            print("\nERROR: La palabra solo puede contener 0 y 1.")
            return False


    # Crear cinta
    cinta = {}

    for i, simbolo in enumerate(palabra):
        cinta[i] = SIMBOLOS[simbolo]


    # Si la palabra es vacía
    if len(palabra) == 0:
        cinta[0] = 0


    # Estado inicial
    estado = 0

    # Estados finales
    estado_aceptacion = 4
    estado_rechazo = 5

    # Cabezal
    cabezal = 0

    pasos = 0


    print("\n" + "=" * 65)
    print("           INICIO DE LA SIMULACION")
    print("=" * 65)

    print(f"Palabra de entrada: {palabra}")

    mostrar_cinta(cinta, cabezal, estado)


    while pasos < max_pasos:

        # ----------------------------------------------------
        # Verificar aceptación
        # ----------------------------------------------------

        if estado == estado_aceptacion:

            print("\n" + "=" * 65)
            print("CADENA ACEPTADA")
            print("=" * 65)

            return True


        # ----------------------------------------------------
        # Verificar rechazo
        # ----------------------------------------------------

        if estado == estado_rechazo:

            print("\n" + "=" * 65)
            print("CADENA RECHAZADA")
            print("=" * 65)

            return False


        # ----------------------------------------------------
        # Leer símbolo
        # ----------------------------------------------------

        simbolo_actual = cinta.get(cabezal, 0)


        # ----------------------------------------------------
        # Buscar transición
        # ----------------------------------------------------

        transicion = buscar_transicion(
            maquina,
            estado,
            simbolo_actual
        )


        if transicion is None:

            print("\nNo existe una transición válida.")
            print("La máquina se detiene.")

            return False


        # ----------------------------------------------------
        # Separar datos de la transición
        # ----------------------------------------------------

        estado_actual = transicion[0]
        simbolo_leido = transicion[1]
        nuevo_estado = transicion[2]
        simbolo_escrito = transicion[3]
        movimiento = transicion[4]


        pasos += 1


        # ----------------------------------------------------
        # Mostrar transición utilizada
        # ----------------------------------------------------

        print("\n" + "-" * 65)

        print(f"Paso {pasos}")

        print(
            f"δ(q{estado_actual}, "
            f"{SIMBOLOS_INV[simbolo_leido]}) "
            f"= "
            f"(q{nuevo_estado}, "
            f"{SIMBOLOS_INV[simbolo_escrito]}, "
            f"{MOVIMIENTOS[movimiento]})"
        )

        print(
            "Codificación binaria:",
            mostrar_codigo_binario(transicion)
        )


        # ----------------------------------------------------
        # Escribir símbolo
        # ----------------------------------------------------

        cinta[cabezal] = simbolo_escrito


        # ----------------------------------------------------
        # Mover cabezal
        # ----------------------------------------------------

        if movimiento == 0:

            cabezal -= 1

        elif movimiento == 1:

            cabezal += 1

        # movimiento == 2 significa no moverse


        # ----------------------------------------------------
        # Cambiar estado
        # ----------------------------------------------------

        estado = nuevo_estado


        # Crear blanco automáticamente
        if cabezal not in cinta:
            cinta[cabezal] = 0


        mostrar_cinta(cinta, cabezal, estado)


    print("\nSe alcanzó el máximo de pasos.")

    return False


3. **Pruebas de funcionamiento**  
   * Probar la simulación con diferentes entradas
   * Documentar los resultados

In [ ]:
# ------------------------------------------------------------
# PRUEBAS
# ------------------------------------------------------------

prueba1 = mtu(maquina, "01")
print("01:", prueba1)

prueba2 = mtu(maquina, "0011")
print("0011:", prueba2)

prueba3 = mtu(maquina, "000111")
print("000111:", prueba3)

prueba4 = mtu(maquina, "001")
print("001:", prueba4)

prueba5 = mtu(maquina, "011")
print("011:", prueba5)

prueba6 = mtu(maquina, "0101")
print("0101:", prueba6)

| Entrada | Resultado esperado | Resultado obtenido |
|---|---|---|
| `01` | Aceptada | Aceptada |
| `0011` | Aceptada | Aceptada |
| `000111` | Aceptada | Aceptada |
| `001` | Rechazada | Rechazada |
| `011` | Rechazada | Rechazada |
| `0101` | Rechazada | Rechazada |

4. **Informe final**  
   * Explicar la codificación utilizada
   * Mostrar ejemplos de ejecución
   * Reflexionar sobre la relación entre la MTU y las computadoras modernas

La Máquina de Turing debe comprobar que la cantidad de ceros sea igual a la cantidad de unos y que todos los ceros aparezcan antes que los unos.
El procedimiento utilizado es el siguiente:
1. La máquina comienza en el estado q0.
2. Busca el primer 0 que todavía no haya sido procesado.
3. Ese 0 se reemplaza por X.
4. La máquina avanza hacia la derecha buscando un 1.
5. Cuando encuentra el 1, lo reemplaza por Y.
6. Luego vuelve hacia la izquierda hasta llegar nuevamente al comienzo de la cadena.
7. El procedimiento se repite mientras existan ceros sin procesar.
8. Cuando ya no quedan ceros, se comprueba que tampoco existan unos sin procesar.
9. Si todos los símbolos fueron procesados correctamente, la máquina llega al estado de aceptación.
10. Si encuentra una diferencia entre la cantidad u orden de los símbolos, llega al estado de rechazo.

El simulador en Python interpreta las transiciones y va modificando la cinta según el símbolo leído y el estado actual.
Durante la ejecución, la máquina utiliza los símbolos X e Y para marcar los ceros y unos que ya fueron procesados.
Las pruebas realizadas demostraron que la máquina acepta correctamente las cadenas que pertenecen al lenguaje y rechaza las que no cumplen con la condición.

Por ejemplo:

0011 -> ACEPTADA

001  -> RECHAZADA

0101 -> RECHAZADA





### Reflexion con las computadoras actuales
La Máquina de Turing Universal se relaciona con las computadoras modernas porque ambas pueden ejecutar diferentes procedimientos a partir de un conjunto de instrucciones.
Esta, puede recibir la descripción de otra máquina y simularla.
Esto es muy similar a una computadora actual, porque no necesita cambiar físicamente para ejecutar diferentes programas.
El mismo hardware puede ejecutar:
- un navegador,
- una calculadora,
- un compilador.

Lo que cambia es el programa que se ejecuta.